In [1]:
!git clone https://github.com/AREEG94FAHAD/TaskComplexityEval-24.git


Cloning into 'TaskComplexityEval-24'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 12 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 2.52 MiB | 3.65 MiB/s, done.


In [2]:
!ls TaskComplexityEval-24


problems_data.jsonl  README.md


In [3]:
!sed -n '1,200p' TaskComplexityEval-24/README.md


# TaskComplexity Dataset [👉Read the paper](https://arxiv.org/abs/2409.20189)

This project addresses the challenge of classifying and assigning programming tasks. A novel dataset containing a total of **4,112 programming tasks** was created by systematically extracting tasks from various websites using web scraping techniques.

## Dataset Overview [download](https://github.com/AREEG94FAHAD/TaskComplexityEval-24/blob/main/problems_data.jsonl)

The **TaskComplexity** dataset provides a comprehensive collection of programming problems, each labeled by classification, enabling the development and evaluation of models based on task difficulty.

### Features

Each entry in the dataset includes the following attributes:

- **Task Title**: The name or title of the programming problem.
- **Problem Description**: A detailed description of the programming task.
- **Input/Output Specification**: A breakdown of the expected input and output for the problem.
- **Examples**: Sample cases to clarify t

In [4]:
!find TaskComplexityEval-24 -type f


TaskComplexityEval-24/.git/index
TaskComplexityEval-24/.git/refs/remotes/origin/HEAD
TaskComplexityEval-24/.git/refs/heads/main
TaskComplexityEval-24/.git/objects/pack/pack-d8d0a211a5277de5e7d6dce81ba1f25b0ff52e14.idx
TaskComplexityEval-24/.git/objects/pack/pack-d8d0a211a5277de5e7d6dce81ba1f25b0ff52e14.pack
TaskComplexityEval-24/.git/logs/refs/remotes/origin/HEAD
TaskComplexityEval-24/.git/logs/refs/heads/main
TaskComplexityEval-24/.git/logs/HEAD
TaskComplexityEval-24/.git/description
TaskComplexityEval-24/.git/packed-refs
TaskComplexityEval-24/.git/HEAD
TaskComplexityEval-24/.git/config
TaskComplexityEval-24/.git/hooks/prepare-commit-msg.sample
TaskComplexityEval-24/.git/hooks/pre-merge-commit.sample
TaskComplexityEval-24/.git/hooks/pre-applypatch.sample
TaskComplexityEval-24/.git/hooks/pre-push.sample
TaskComplexityEval-24/.git/hooks/update.sample
TaskComplexityEval-24/.git/hooks/pre-commit.sample
TaskComplexityEval-24/.git/hooks/commit-msg.sample
TaskComplexityEval-24/.git/hooks/pus

In [5]:
import pandas as pd

df = pd.read_json(
    "/content/TaskComplexityEval-24/problems_data.jsonl",
    lines=True
)

df.head()


,title,description,input_description,output_description,sample_io,problem_class,problem_score,url
0,Uuu,Unununium (Uuu) was the name of the chemical\n...,The input consists of one line with two intege...,The output consists of $M$ lines where the $i$...,"[{'input': '7 10', 'output': '1 2 2 3 1 3 3 4 ...",hard,9.7,https://open.kattis.com/problems/uuu
1,House Building,A number of eccentrics from central New York h...,"The input consists of $10$ test cases, which a...",Print $K$ lines with\n the positions of the...,"[{'input': '0 2 3 2 50 60 50 30 50 40', 'outpu...",hard,9.7,https://open.kattis.com/problems/husbygge
2,Mario or Luigi,Mario and Luigi are playing a game where they ...,,,"[{'input': '', 'output': ''}]",hard,9.6,https://open.kattis.com/problems/marioorluigi
3,The Wire Ghost,Žofka is bending a copper wire. She starts wit...,The first line contains two integers $L$ and $...,The output consists of a single line consistin...,"[{'input': '4 3 3 C 2 C 1 C', 'output': 'GHOST...",hard,9.6,https://open.kattis.com/problems/thewireghost
4,Barking Up The Wrong Tree,"Your dog Spot is let loose in the park. Well, ...",The first line of input consists of two intege...,Write a single line containing the length need...,"[{'input': '2 0 10 0 10 10', 'output': '14.14'...",hard,9.6,https://open.kattis.com/problems/barktree


In [6]:
df.columns


Index(['title', 'description', 'input_description', 'output_description',
       'sample_io', 'problem_class', 'problem_score', 'url'],
      dtype='object')

In [7]:
df["full_text"] = (
    df["title"].fillna("") + " " +
    df["description"].fillna("") + " " +
    df["input_description"].fillna("") + " " +
    df["output_description"].fillna("")
)


In [8]:
df[["problem_class", "problem_score"]].head()


,problem_class,problem_score
0,hard,9.7
1,hard,9.7
2,hard,9.6
3,hard,9.6
4,hard,9.6


In [9]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error

from scipy.sparse import hstack
import joblib


In [10]:
keywords = ["dp", "graph", "tree", "recursion", "greedy", "bitmask"]

def extract_features(text):
    feats = {}
    feats["text_length"] = len(text)
    feats["math_symbols"] = len(re.findall(r"[+\-*/=<>()]", text))
    for kw in keywords:
        feats[f"kw_{kw}"] = text.lower().count(kw)
    return feats


In [11]:
feature_df = df["full_text"].apply(extract_features).apply(pd.Series)


In [12]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2)
)

X_text = tfidf.fit_transform(df["full_text"])


In [13]:
X = hstack([X_text, feature_df.values])


In [14]:
y_class = df["problem_class"]


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_class, test_size=0.2, random_state=42
)


In [16]:
clf = LinearSVC()
clf.fit(X_train, y_train)


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


LinearSVC()

In [17]:
y_pred = clf.predict(X_test)

print("Classification Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Classification Accuracy: 0.520048602673147
Confusion Matrix:
 [[  4 132   0]
 [  1 424   0]
 [  0 262   0]]


In [18]:
y_score = df["problem_score"]


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_score, test_size=0.2, random_state=42
)


In [20]:
reg = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

reg.fit(X_train, y_train)


RandomForestRegressor(n_estimators=200, random_state=42)

In [21]:
preds = reg.predict(X_test)

print("MAE:", mean_absolute_error(y_test, preds))
print("RMSE:", np.sqrt(mean_squared_error(y_test, preds)))


MAE: 1.6973620899149455
RMSE: 2.0485177050664802


In [22]:
joblib.dump(clf, "classifier.pkl")
joblib.dump(reg, "regressor.pkl")
joblib.dump(tfidf, "tfidf.pkl")


['tfidf.pkl']

In [24]:
pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 40.4 MB/s eta 0:00:00


In [25]:
import streamlit as st
import joblib
import numpy as np
import re
from scipy.sparse import hstack

clf = joblib.load("classifier.pkl")
reg = joblib.load("regressor.pkl")
tfidf = joblib.load("tfidf.pkl")

st.title("AutoJudge – Programming Problem Difficulty Predictor")

desc = st.text_area("Problem Description")
inp = st.text_area("Input Description")
out = st.text_area("Output Description")

def extract_features(text):
    keywords = ["dp", "graph", "tree", "recursion", "greedy", "bitmask"]
    feats = []
    feats.append(len(text))
    feats.append(len(re.findall(r"[+\-*/=<>()]", text)))
    for kw in keywords:
        feats.append(text.lower().count(kw))
    return np.array(feats).reshape(1, -1)

if st.button("Predict"):
    full_text = desc + " " + inp + " " + out
    X_text = tfidf.transform([full_text])
    X_feat = extract_features(full_text)
    X_final = hstack([X_text, X_feat])

    st.success(f"Predicted Difficulty Class: {clf.predict(X_final)[0]}")
    st.success(f"Predicted Difficulty Score: {reg.predict(X_final)[0]:.2f}")


2025-12-31 08:02:39.198 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-31 08:02:39.961 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-12-31 08:02:39.962 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-31 08:02:39.963 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-31 08:02:39.968 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-31 08:02:39.970 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-31 08:02:39.980 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-31 08:02:39.983 Thread 'MainThread': mi

In [26]:
%%writefile app.py
import streamlit as st
import joblib
import numpy as np
import re
from scipy.sparse import hstack

# Load trained models
clf = joblib.load("classifier.pkl")
reg = joblib.load("regressor.pkl")
tfidf = joblib.load("tfidf.pkl")

st.title("AutoJudge – Programming Problem Difficulty Predictor")

desc = st.text_area("Problem Description")
inp = st.text_area("Input Description")
out = st.text_area("Output Description")

def extract_features(text):
    keywords = ["dp", "graph", "tree", "recursion", "greedy", "bitmask"]
    feats = []
    feats.append(len(text))
    feats.append(len(re.findall(r"[+\-*/=<>()]", text)))
    for kw in keywords:
        feats.append(text.lower().count(kw))
    return np.array(feats).reshape(1, -1)

if st.button("Predict"):
    full_text = desc + " " + inp + " " + out
    X_text = tfidf.transform([full_text])
    X_feat = extract_features(full_text)
    X_final = hstack([X_text, X_feat])

    st.success(f"Predicted Difficulty Class: {clf.predict(X_final)[0]}")
    st.success(f"Predicted Difficulty Score: {reg.predict(X_final)[0]:.2f}")


Writing app.py


In [27]:
!ls


app.py		regressor.pkl  TaskComplexityEval-24
classifier.pkl	sample_data    tfidf.pkl
